# Memahami LLM melalui kode: baseline

Notebook mandiri untuk pembaca dengan dasar matematika. Jalankan dari atas ke bawah
pada kernel Python dengan PyTorch terpasang. CPU cukup. Semua data, bobot, dan hasil
berada di memori; hasil muncul di bawah sel ketika dijalankan. Sel definisi memang
belum menghasilkan output. Notebook yang dibagikan menyimpan output kosong.

Baca urutannya: definisi model, data karakter, attention, loss, gradien, sepuluh
langkah latihan, lalu generasi. Buka tabel AntiVibe di dekat kode untuk melacak nama
variabel. Jika ada perubahan definisi, restart kernel lalu Run all.

$B$ = jumlah contoh, $T$ = panjang urutan, $C$ = dimensi embedding,
$V$ = banyak token, $H=4C$ = lebar MLP. Tensor adalah array multidimensi;
`[B,T,C]` dapat dipandang sebagai $BT$ vektor di $\mathbb{R}^C$.

Referensi konteks: [tutorial penulis](https://fareedkhan-dev.github.io/train-llm-from-scratch/).
Kode berikut diambil dari repositori lokal; varian sparse merupakan eksperimen repo ini.


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `kernel` | `F` | Fungsi tensor neural network tanpa objek layer, misalnya linear dan cross entropy. |
| `kernel` | `Tensor` | Tipe tensor PyTorch yang dipakai dalam anotasi fungsi. |
| `kernel` | `checkpoint` | Modul checkpoint aktivasi; menghitung ulang forward untuk menghemat memori. |
| `kernel` | `copy` | Modul penyalin objek; deepcopy membuat parameter pembanding independen. |
| `kernel` | `math` | Modul matematika Python untuk operasi skalar. |
| `kernel` | `nn` | Modul torch.nn untuk layer dan registrasi parameter model. |
| `kernel` | `torch` | Modul PyTorch: tensor, diferensiasi otomatis, dan optimizer. |

</details>


In [ ]:
import copy
import math
import torch
from torch import Tensor
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint

torch.set_num_threads(1)
print('PyTorch:', torch.__version__, '| laboratorium CPU')


### Membaca `MLP`

MLP mengubah tiap vektor token secara terpisah: $h=\max(0,xW_1^\top+b_1)$, lalu $y=hW_2^\top+b_2$. $W_1\in\mathbb R^{4C\times C}$ dan $W_2\in\mathbb R^{C\times4C}$. ReLU memberi nonlinieritas; tanpa itu dua transformasi affine dapat digabung menjadi satu. Alternatifnya, GELU memberi transisi yang lebih halus.

<details><summary>Latihan membaca kode</summary>

Sebelum menjalankan, telusuri satu input sampai `return` (atau `yield`).
Apa yang tetap dan apa yang berubah pada bentuk tensor atau objeknya?
Cocokkan dengan tabel nama di bawah dan hasil demo setelah definisi model.
</details>


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `MLP.__init__` | `hidden` | Aktivasi hidden tensor [...,H], atau atribut layer Linear(C,H); H=4C pada MLP dense. |
| `MLP.__init__` | `n_embed` | Integer C: lebar representasi satu token. |
| `MLP.__init__` | `proj` | Layer Linear(H,C) yang mengembalikan fitur hidden ke lebar embedding. |
| `MLP.__init__` | `relu` | Layer ReLU yang mengganti nilai negatif dengan nol. |
| `MLP.__init__` | `self` | Objek instance kelas saat ini; atributnya bertahan antar pemanggilan metode. |
| `MLP.forward` | `self` | Objek instance kelas saat ini; atributnya bertahan antar pemanggilan metode. |
| `MLP.forward` | `x` | Tensor fitur input/intermediat [...,C]; dalam MLP dapat menjadi [...,4C] setelah hidden. |
| `MLP.forward_embedding` | `self` | Objek instance kelas saat ini; atributnya bertahan antar pemanggilan metode. |
| `MLP.forward_embedding` | `x` | Tensor fitur input/intermediat [...,C]; dalam MLP dapat menjadi [...,4C] setelah hidden. |
| `MLP.project_embedding` | `self` | Objek instance kelas saat ini; atributnya bertahan antar pemanggilan metode. |
| `MLP.project_embedding` | `x` | Tensor fitur input/intermediat [...,C]; dalam MLP dapat menjadi [...,4C] setelah hidden. |

| Fungsi atau kelas | Peran |
| :--- | :--- |
| `MLP` | MLP mengubah tiap vektor token secara terpisah: $h=\max(0,xW_1^\top+b_1)$, lalu $y=hW_2^\top+b_2$. |
| `MLP.__init__` | Membangun layer dan buffer, memeriksa konfigurasi; tidak mengembalikan tensor. |
| `MLP.forward` | Menghitung keluaran layer/model dari input. Dipanggil otomatis oleh objek(input); bentuk dan rumus dijelaskan di atas. |
| `MLP.forward_embedding` | Mengembalikan aktivasi hidden internal; pada Block/Transformer juga residual. Ini jalur inspeksi, bukan forward logits. |
| `MLP.project_embedding` | Memproyeksikan aktivasi hidden lebar 4C ke C melalui layer projection. |

</details>


In [ ]:
class MLP(nn.Module):
    """
    A simple Multi-Layer Perceptron with one hidden layer.

    This module is used within the Transformer block for feed-forward processing.
    It expands the input embedding size, applies a ReLU activation, and then projects it back
    to the original embedding size.

    Args:
        n_embed (int): The dimensionality of the input embedding.
    """
    def __init__(self, n_embed: int) -> None:
        """
        Initializes the MLP module.

        Args:
            n_embed (int): The dimensionality of the input embedding.
        """
        super().__init__()
        self.hidden = nn.Linear(n_embed, 4 * n_embed)  # Linear layer to expand embedding size
        self.relu = nn.ReLU()                        # ReLU activation function
        self.proj = nn.Linear(4 * n_embed, n_embed)  # Linear layer to project back to original size

    def forward(self, x: Tensor) -> Tensor:
        """
        Forward pass through the MLP.

        Args:
            x (torch.Tensor): Input tensor of shape (B, T, C), where B is batch size,
                              T is sequence length, and C is embedding size.

        Returns:
            torch.Tensor: Output tensor of the same shape as the input.
        """
        x = self.forward_embedding(x)
        x = self.project_embedding(x)
        return x

    def forward_embedding(self, x: Tensor) -> Tensor:
        """
        Applies the hidden linear layer followed by ReLU activation.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Output after the hidden layer and ReLU.
        """
        x = self.relu(self.hidden(x))
        return x

    def project_embedding(self, x: Tensor) -> Tensor:
        """
        Applies the projection linear layer.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Output after the projection layer.
        """
        x = self.proj(x)
        return x


### Membaca `Head`

Satu head mencampur informasi antarposisi. $Q=XW_Q$, $K=XW_K$, $V=XW_V$, lalu $A=\operatorname{softmax}(QK^\top/\sqrt d+M)$ dan $O=AV$. Di sini $d$ lebar satu head; $M_{ij}=-\infty$ jika $j>i$ dan nol selainnya. Softmax bekerja per baris. Pembagi $\sqrt d$ mengendalikan skala dot product. Mask mencegah bocornya jawaban masa depan. [Paper attention](https://arxiv.org/abs/1706.03762).

<details><summary>Latihan membaca kode</summary>

Sebelum menjalankan, telusuri satu input sampai `return` (atau `yield`).
Apa yang tetap dan apa yang berubah pada bentuk tensor atau objeknya?
Cocokkan dengan tabel nama di bawah dan hasil demo setelah definisi model.
</details>


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `Head.__init__` | `context_length` | Integer batas panjang konteks dan ukuran embedding posisi/mask causal. |
| `Head.__init__` | `head_size` | Integer lebar fitur satu head, C dibagi jumlah head. |
| `Head.__init__` | `key` | Layer proyeksi fitur menjadi key/query/value satu head; value lokal di cleanup adalah tensor state Adam. |
| `Head.__init__` | `n_embed` | Integer C: lebar representasi satu token. |
| `Head.__init__` | `query` | Layer proyeksi fitur menjadi key/query/value satu head; value lokal di cleanup adalah tensor state Adam. |
| `Head.__init__` | `self` | Objek instance kelas saat ini; atributnya bertahan antar pemanggilan metode. |
| `Head.__init__` | `tril` | Buffer mask segitiga bawah [context_length,context_length], bukan parameter terlatih. |
| `Head.__init__` | `value` | Layer proyeksi fitur menjadi key/query/value satu head; value lokal di cleanup adalah tensor state Adam. |
| `Head.forward` | `B` | Integer jumlah contoh dalam batch. |
| `Head.forward` | `C` | Integer dimensi terakhir tensor; pada attention adalah lebar input, pada Transformer.forward adalah V (logits). |
| `Head.forward` | `T` | Integer panjang urutan pada pemanggilan ini. |
| `Head.forward` | `attn_weights` | Tensor [B,T,T] skor attention yang kemudian dimask dan dinormalisasi softmax. |
| `Head.forward` | `head_size` | Integer lebar fitur satu head, C dibagi jumlah head. |
| `Head.forward` | `k` | Tensor key/query/value [B,T,d] pada attention; k atribut TopKMLP adalah jumlah neuron terpilih. |
| `Head.forward` | `out` | Tensor hasil attention atau MLP, dengan dimensi akhir sesuai proyeksi yang dijalankan. |
| `Head.forward` | `q` | Tensor key/query/value [B,T,d] pada attention; k atribut TopKMLP adalah jumlah neuron terpilih. |
| `Head.forward` | `scale_factor` | Float faktor 1/sqrt(d) untuk mengendalikan skala dot product. |
| `Head.forward` | `self` | Objek instance kelas saat ini; atributnya bertahan antar pemanggilan metode. |
| `Head.forward` | `v` | Tensor key/query/value [B,T,d] pada attention; k atribut TopKMLP adalah jumlah neuron terpilih. |
| `Head.forward` | `x` | Tensor fitur input/intermediat [...,C]; dalam MLP dapat menjadi [...,4C] setelah hidden. |

| Fungsi atau kelas | Peran |
| :--- | :--- |
| `Head` | Satu head mencampur informasi antarposisi. |
| `Head.__init__` | Membangun layer dan buffer, memeriksa konfigurasi; tidak mengembalikan tensor. |
| `Head.forward` | Menghitung keluaran layer/model dari input. Dipanggil otomatis oleh objek(input); bentuk dan rumus dijelaskan di atas. |

</details>


In [ ]:
class Head(nn.Module):
    """
    A single attention head.

    This module calculates attention scores and applies them to the values.
    It includes key, query, and value projections, and uses causal masking
    to prevent attending to future tokens.

    Args:
        head_size (int): The dimensionality of the key, query, and value projections.
        n_embed (int): The dimensionality of the input embedding.
        context_length (int): The maximum length of the input sequence, used for causal masking.
    """
    def __init__(self, head_size: int, n_embed: int, context_length: int) -> None:
        """
        Initializes the attention head.

        Args:
            head_size (int): The dimensionality of the key, query, and value projections.
            n_embed (int): The dimensionality of the input embedding.
            context_length (int): The maximum length of the input sequence.
        """
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)   # Key projection
        self.query = nn.Linear(n_embed, head_size, bias=False) # Query projection
        self.value = nn.Linear(n_embed, head_size, bias=False) # Value projection
        # Lower triangular matrix for causal masking
        self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the attention head.

        Args:
            x (torch.Tensor): Input tensor of shape (B, T, C).

        Returns:
            torch.Tensor: Output tensor after applying attention.
        """
        B, T, C = x.shape
        head_size = self.key.out_features
        k = self.key(x)     # (B, T, head_size)
        q = self.query(x)   # (B, T, head_size)
        scale_factor = 1 / math.sqrt(head_size)
        # Calculate attention weights: (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
        attn_weights = q @ k.transpose(-2, -1) * scale_factor
        # Apply causal masking
        attn_weights = attn_weights.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        attn_weights = F.softmax(attn_weights, dim=-1)
        v = self.value(x)   # (B, T, head_size)
        # Apply attention weights to values
        out = attn_weights @ v # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
        return out


### Membaca `MultiHeadAttention`

Beberapa head membaca proyeksi yang berbeda. Hasilnya digabung pada sumbu fitur lalu diproyeksikan ke $C$: $O=\operatorname{concat}(O_1,\ldots,O_h)W_O$. Pilih $C$ habis dibagi jumlah head; konfigurasi demo memakai $16/2=8$.

<details><summary>Latihan membaca kode</summary>

Sebelum menjalankan, telusuri satu input sampai `return` (atau `yield`).
Apa yang tetap dan apa yang berubah pada bentuk tensor atau objeknya?
Cocokkan dengan tabel nama di bawah dan hasil demo setelah definisi model.
</details>


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `MultiHeadAttention.__init__` | `_` | Nilai atau indeks yang sengaja tidak digunakan. |
| `MultiHeadAttention.__init__` | `context_length` | Integer batas panjang konteks dan ukuran embedding posisi/mask causal. |
| `MultiHeadAttention.__init__` | `heads` | ModuleList berisi seluruh head; PyTorch mendaftarkan parameternya secara otomatis. |
| `MultiHeadAttention.__init__` | `n_embed` | Integer C: lebar representasi satu token. |
| `MultiHeadAttention.__init__` | `n_head` | Integer jumlah head attention; C harus habis dibagi jumlah ini. |
| `MultiHeadAttention.__init__` | `proj` | Layer Linear(H,C) yang mengembalikan fitur hidden ke lebar embedding. |
| `MultiHeadAttention.__init__` | `self` | Objek instance kelas saat ini; atributnya bertahan antar pemanggilan metode. |
| `MultiHeadAttention.forward` | `h` | Objek head saat iterasi list comprehension. |
| `MultiHeadAttention.forward` | `self` | Objek instance kelas saat ini; atributnya bertahan antar pemanggilan metode. |
| `MultiHeadAttention.forward` | `x` | Tensor fitur input/intermediat [...,C]; dalam MLP dapat menjadi [...,4C] setelah hidden. |

| Fungsi atau kelas | Peran |
| :--- | :--- |
| `MultiHeadAttention` | Beberapa head membaca proyeksi yang berbeda. |
| `MultiHeadAttention.__init__` | Membangun layer dan buffer, memeriksa konfigurasi; tidak mengembalikan tensor. |
| `MultiHeadAttention.forward` | Menghitung keluaran layer/model dari input. Dipanggil otomatis oleh objek(input); bentuk dan rumus dijelaskan di atas. |

</details>


In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention module.

    This module combines multiple attention heads in parallel. The outputs of each head
    are concatenated and passed through a final linear projection to form the output.

    Args:
        n_head (int): The number of parallel attention heads.
        n_embed (int): The dimensionality of the input embedding.
        context_length (int): The maximum length of the input sequence.
    """
    def __init__(self, n_head: int, n_embed: int, context_length: int) -> None:
        """
        Initializes the multi-head attention module.

        Args:
            n_head (int): The number of parallel attention heads.
            n_embed (int): The dimensionality of the input embedding.
            context_length (int): The maximum length of the input sequence.
        """
        super().__init__()
        self.heads = nn.ModuleList([Head(n_embed // n_head, n_embed, context_length) for _ in range(n_head)])
        self.proj = nn.Linear(n_embed, n_embed)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the multi-head attention.

        Args:
            x (torch.Tensor): Input tensor of shape (B, T, C).

        Returns:
            torch.Tensor: Output tensor after concatenating the heads and applying the output projection.
        """
        # Concatenate the output of each head along the last dimension (C)
        x = torch.cat([h(x) for h in self.heads], dim=-1)
        # Apply final linear projection
        x = self.proj(x)
        return x


### Membaca `Block`

Blok memakai residual: $u=x+\operatorname{Attn}(\operatorname{LN}(x))$, $y=u+\operatorname{MLP}(\operatorname{LN}(u))$. LN adalah LayerNorm pada fitur satu token. Residual menyediakan jalur penjumlahan langsung untuk informasi dan gradien. Implementasi ini menormalisasi sebelum sublayer; arsitektur lain menaruh normalisasi sesudahnya.

<details><summary>Latihan membaca kode</summary>

Sebelum menjalankan, telusuri satu input sampai `return` (atau `yield`).
Apa yang tetap dan apa yang berubah pada bentuk tensor atau objeknya?
Cocokkan dengan tabel nama di bawah dan hasil demo setelah definisi model.
</details>


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `Block.__init__` | `attn` | Objek MultiHeadAttention dalam satu blok. |
| `Block.__init__` | `context_length` | Integer batas panjang konteks dan ukuran embedding posisi/mask causal. |
| `Block.__init__` | `ln1` | LayerNorm atas dimensi fitur; ln1 sebelum attention, ln2 sebelum MLP, layer_norm di akhir backbone. |
| `Block.__init__` | `ln2` | LayerNorm atas dimensi fitur; ln1 sebelum attention, ln2 sebelum MLP, layer_norm di akhir backbone. |
| `Block.__init__` | `mlp` | Objek MLP blok, dense atau varian sparse sesuai notebook. |
| `Block.__init__` | `n_embed` | Integer C: lebar representasi satu token. |
| `Block.__init__` | `n_head` | Integer jumlah head attention; C harus habis dibagi jumlah ini. |
| `Block.__init__` | `self` | Objek instance kelas saat ini; atributnya bertahan antar pemanggilan metode. |
| `Block.forward` | `self` | Objek instance kelas saat ini; atributnya bertahan antar pemanggilan metode. |
| `Block.forward` | `x` | Tensor fitur input/intermediat [...,C]; dalam MLP dapat menjadi [...,4C] setelah hidden. |
| `Block.forward_embedding` | `res` | Tensor [B,T,C] jalur residual yang dijumlahkan dengan keluaran sublayer. |
| `Block.forward_embedding` | `self` | Objek instance kelas saat ini; atributnya bertahan antar pemanggilan metode. |
| `Block.forward_embedding` | `x` | Tensor fitur input/intermediat [...,C]; dalam MLP dapat menjadi [...,4C] setelah hidden. |

| Fungsi atau kelas | Peran |
| :--- | :--- |
| `Block` | Blok memakai residual: $u=x+\operatorname{Attn}(\operatorname{LN}(x))$, $y=u+\operatorname{MLP}(\operatorname{LN}(u))$. |
| `Block.__init__` | Membangun layer dan buffer, memeriksa konfigurasi; tidak mengembalikan tensor. |
| `Block.forward` | Menghitung keluaran layer/model dari input. Dipanggil otomatis oleh objek(input); bentuk dan rumus dijelaskan di atas. |
| `Block.forward_embedding` | Mengembalikan aktivasi hidden internal; pada Block/Transformer juga residual. Ini jalur inspeksi, bukan forward logits. |

</details>


In [ ]:
class Block(nn.Module):
    """
    A single Transformer block.

    This block consists of a multi-head attention layer followed by an MLP,
    with layer normalization and residual connections.

    Args:
        n_head (int): The number of attention heads in the multi-head attention layer.
        n_embed (int): The dimensionality of the input embedding.
        context_length (int): The maximum length of the input sequence.
    """
    def __init__(self, n_head: int, n_embed: int, context_length: int) -> None:
        """
        Initializes the Transformer block.

        Args:
            n_head (int): The number of attention heads.
            n_embed (int): The dimensionality of the embedding space.
            context_length (int): The maximum sequence length.
        """
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embed)
        self.attn = MultiHeadAttention(n_head, n_embed, context_length)
        self.ln2 = nn.LayerNorm(n_embed)
        self.mlp = MLP(n_embed)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the Transformer block.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Output tensor after the block.
        """
        # Apply multi-head attention with residual connection
        x = x + self.attn(self.ln1(x))
        # Apply MLP with residual connection
        x = x + self.mlp(self.ln2(x))
        return x

    def forward_embedding(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass focusing on the embedding and attention parts.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            tuple: A tuple containing the output after MLP embedding and the residual.
        """
        res = x + self.attn(self.ln1(x))
        x = self.mlp.forward_embedding(self.ln2(res))
        return x, res


### Membaca `Transformer`

Token ID memilih baris embedding, bukan mengalikan nilai ID sebagai bilangan bermakna. Embedding posisi ditambahkan untuk membedakan urutan. Setelah blok, linear head menghasilkan logits $z\in\mathbb R^V$ dan $p_j=e^{z_j}/\sum_k e^{z_k}$. Loss next-token adalah $L=-\frac1{BT}\sum_{b,t}\log p_{b,t,y_{b,t}}$. `forward` mengembalikan pasangan `(logits, loss)`; loss `None` bila target tidak diberikan.

`forward_hidden` adalah jalur representasi akhir `[B,T,C]`. Metode lama `forward_embedding` mengeluarkan aktivasi internal lebar $4C$; jangan merangkainya untuk lebih dari satu blok karena blok berikut mengharapkan $C$. Demo tidak memakai metode lama itu. Gradient checkpointing berarti menghitung ulang aktivasi saat backward, tidak menyimpan berkas checkpoint. Generasi memakai sampling; argmax adalah alternatif deterministik.

<details><summary>Latihan membaca kode</summary>

Sebelum menjalankan, telusuri satu input sampai `return` (atau `yield`).
Apa yang tetap dan apa yang berubah pada bentuk tensor atau objeknya?
Cocokkan dengan tabel nama di bawah dan hasil demo setelah definisi model.
</details>


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `Transformer.__init__` | `N_BLOCKS` | Integer jumlah blok Transformer yang disusun berurutan. |
| `Transformer.__init__` | `_` | Nilai atau indeks yang sengaja tidak digunakan. |
| `Transformer.__init__` | `attn_blocks` | ModuleList berisi blok-blok Transformer berurutan. |
| `Transformer.__init__` | `context_length` | Integer batas panjang konteks dan ukuran embedding posisi/mask causal. |
| `Transformer.__init__` | `gradient_checkpointing` | Boolean untuk menghitung ulang aktivasi saat backward; demo membiarkannya False. |
| `Transformer.__init__` | `layer_norm` | LayerNorm atas dimensi fitur; ln1 sebelum attention, ln2 sebelum MLP, layer_norm di akhir backbone. |
| `Transformer.__init__` | `lm_head` | Linear(C,V): proyeksi fitur terakhir menjadi logits vocabulary. |
| `Transformer.__init__` | `n_embed` | Integer C: lebar representasi satu token. |
| `Transformer.__init__` | `n_head` | Integer jumlah head attention; C harus habis dibagi jumlah ini. |
| `Transformer.__init__` | `pos_idxs` | Buffer integer indeks posisi 0 sampai context_length-1. |
| `Transformer.__init__` | `position_embed` | Embedding posisi terlatih [context_length,C]. |
| `Transformer.__init__` | `self` | Objek instance kelas saat ini; atributnya bertahan antar pemanggilan metode. |
| `Transformer.__init__` | `token_embed` | Embedding terlatih [V,C] yang memetakan ID token ke vektor. |
| `Transformer.__init__` | `vocab_size` | Integer V: jumlah token berbeda yang dapat diprediksi. |
| `Transformer._pre_attn_pass` | `B` | Integer jumlah contoh dalam batch. |
| `Transformer._pre_attn_pass` | `T` | Integer panjang urutan pada pemanggilan ini. |
| `Transformer._pre_attn_pass` | `idx` | Tensor integer [B,T] berisi ID token; bukan embedding. |
| `Transformer._pre_attn_pass` | `pos_embedding` | Tensor embedding posisi [T,C], disiarkan ke seluruh batch saat penjumlahan. |
| `Transformer._pre_attn_pass` | `self` | Objek instance kelas saat ini; atributnya bertahan antar pemanggilan metode. |
| `Transformer._pre_attn_pass` | `tok_embedding` | Tensor embedding token [B,T,C]. |
| `Transformer.forward` | `B` | Integer jumlah contoh dalam batch. |
| `Transformer.forward` | `C` | Integer dimensi terakhir tensor; pada attention adalah lebar input, pada Transformer.forward adalah V (logits). |
| `Transformer.forward` | `T` | Integer panjang urutan pada pemanggilan ini. |
| `Transformer.forward` | `flat_logits` | Logits [B*T,V] untuk cross entropy. |
| `Transformer.forward` | `idx` | Tensor integer [B,T] berisi ID token; bukan embedding. |
| `Transformer.forward` | `logits` | Skor sebelum softmax [B,T,V]; dalam DPO berupa selisih log-ratio [B], saat generasi diiris menjadi [B,V]. |
| `Transformer.forward` | `loss` | Tensor skalar objective; nama awalan menunjukkan CE, pembanding dense, SFT, reward, DPO, PPO, atau GRPO. |
| `Transformer.forward` | `self` | Objek instance kelas saat ini; atributnya bertahan antar pemanggilan metode. |
| `Transformer.forward` | `targets` | Tensor integer target next-token; [B,T] sebelum flatten atau [B*T] untuk cross entropy. |
| `Transformer.forward` | `x` | Tensor fitur input/intermediat [...,C]; dalam MLP dapat menjadi [...,4C] setelah hidden. |
| `Transformer.forward_embedding` | `block` | Objek blok Transformer saat iterasi atau blok pertama dalam demo. |
| `Transformer.forward_embedding` | `idx` | Tensor integer [B,T] berisi ID token; bukan embedding. |
| `Transformer.forward_embedding` | `residual` | Tensor [B,T,C] jalur residual yang dijumlahkan dengan keluaran sublayer. |
| `Transformer.forward_embedding` | `self` | Objek instance kelas saat ini; atributnya bertahan antar pemanggilan metode. |
| `Transformer.forward_embedding` | `x` | Tensor fitur input/intermediat [...,C]; dalam MLP dapat menjadi [...,4C] setelah hidden. |
| `Transformer.forward_hidden` | `block` | Objek blok Transformer saat iterasi atau blok pertama dalam demo. |
| `Transformer.forward_hidden` | `idx` | Tensor integer [B,T] berisi ID token; bukan embedding. |
| `Transformer.forward_hidden` | `self` | Objek instance kelas saat ini; atributnya bertahan antar pemanggilan metode. |
| `Transformer.forward_hidden` | `x` | Tensor fitur input/intermediat [...,C]; dalam MLP dapat menjadi [...,4C] setelah hidden. |
| `Transformer.generate` | `_` | Nilai atau indeks yang sengaja tidak digunakan. |
| `Transformer.generate` | `idx` | Tensor integer [B,T] berisi ID token; bukan embedding. |
| `Transformer.generate` | `idx_cond` | Tensor ID konteks terakhir, panjangnya dibatasi context_length. |
| `Transformer.generate` | `idx_next` | Tensor integer [B,1] hasil sampling token berikutnya. |
| `Transformer.generate` | `logits` | Skor sebelum softmax [B,T,V]; dalam DPO berupa selisih log-ratio [B], saat generasi diiris menjadi [B,V]. |
| `Transformer.generate` | `max_new_tokens` | Integer jumlah token tambahan yang di-sampling. |
| `Transformer.generate` | `probs` | Tensor probabilitas next-token [B,V] setelah softmax. |
| `Transformer.generate` | `self` | Objek instance kelas saat ini; atributnya bertahan antar pemanggilan metode. |

| Fungsi atau kelas | Peran |
| :--- | :--- |
| `Transformer` | Token ID memilih baris embedding, bukan mengalikan nilai ID sebagai bilangan bermakna. |
| `Transformer.__init__` | Membangun layer dan buffer, memeriksa konfigurasi; tidak mengembalikan tensor. |
| `Transformer._pre_attn_pass` | Mengembalikan jumlah embedding token dan posisi [B,T,C]. |
| `Transformer.forward_hidden` | Menjalankan seluruh blok dan normalisasi akhir; hasil [B,T,C]. |
| `Transformer.forward` | Menghitung keluaran layer/model dari input. Dipanggil otomatis oleh objek(input); bentuk dan rumus dijelaskan di atas. |
| `Transformer.forward_embedding` | Mengembalikan aktivasi hidden internal; pada Block/Transformer juga residual. Ini jalur inspeksi, bukan forward logits. |
| `Transformer.generate` | Mengambil sampel satu token berulang kali; hasil ID prompt ditambah max_new_tokens. |

</details>


In [ ]:
class Transformer(nn.Module):
    """
    The main Transformer model.

    This class combines token and position embeddings with a sequence of Transformer blocks
    and a final linear layer for language modeling.

    Args:
        n_head (int): The number of attention heads in each transformer block.
        n_embed (int): The dimensionality of the embedding space.
        context_length (int): The maximum length of the input sequence.
        vocab_size (int): The size of the vocabulary.
        N_BLOCKS (int): The number of transformer blocks in the model.
    """
    def __init__(self, n_head: int, n_embed: int, context_length: int, vocab_size: int, N_BLOCKS: int) -> None:
        """
        Initializes the Transformer model.

        Args:
            n_head (int): Number of attention heads.
            n_embed (int): Embedding dimension.
            context_length (int): Maximum sequence length.
            vocab_size (int): Size of the vocabulary.
            N_BLOCKS (int): Number of transformer blocks.
        """
        super().__init__()
        self.context_length = context_length
        self.N_BLOCKS = N_BLOCKS
        # Opt-in activation (gradient) checkpointing; off by default so behaviour and
        # numerics are unchanged unless a caller explicitly turns it on (e.g. the
        # pretraining script's --grad-checkpointing flag to save VRAM). See issue #5.
        self.gradient_checkpointing = False
        self.token_embed = nn.Embedding(vocab_size, n_embed)
        self.position_embed = nn.Embedding(context_length, n_embed)
        self.attn_blocks = nn.ModuleList([Block(n_head, n_embed, context_length) for _ in range(N_BLOCKS)])
        self.layer_norm = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)
        self.register_buffer('pos_idxs', torch.arange(context_length))

    def _pre_attn_pass(self, idx: torch.Tensor) -> torch.Tensor:
        """
        Combines token and position embeddings.

        Args:
            idx (torch.Tensor): Input token indices.

        Returns:
            torch.Tensor: Sum of token and position embeddings.
        """
        B, T = idx.shape
        tok_embedding = self.token_embed(idx)
        pos_embedding = self.position_embed(self.pos_idxs[:T])
        return tok_embedding + pos_embedding

    def forward_hidden(self, idx: torch.Tensor) -> torch.Tensor:
        """
        Run the backbone and return the final hidden states AFTER the final layer norm.

        This is exactly the tensor that ``lm_head`` consumes, so it is the right
        representation for auxiliary heads added during post-training (a scalar value
        head for PPO, a scalar reward head for the reward model). Keeping it as a
        separate method lets those heads reuse the backbone without duplicating the
        forward logic or rewriting ``forward``.

        Args:
            idx (torch.Tensor): Input token indices, shape (B, T).

        Returns:
            torch.Tensor: Final hidden states, shape (B, T, n_embed).
        """
        x = self._pre_attn_pass(idx)
        for block in self.attn_blocks:
            if self.gradient_checkpointing and self.training:
                # Recompute block activations in backward instead of storing them,
                # trading compute for a large activation-memory saving. use_reentrant=False
                # is the modern, correct variant.
                x = checkpoint.checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
        return self.layer_norm(x)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor = None) -> tuple[torch.Tensor, torch.Tensor | None]:
        """
        Forward pass through the Transformer.

        Args:
            idx (torch.Tensor): Input token indices.
            targets (torch.Tensor, optional): Target token indices for loss calculation. Defaults to None.

        Returns:
            tuple: Logits and loss (if targets are provided).
        """
        x = self.forward_hidden(idx)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            # reshape (not view): targets come from a non-contiguous slice of the
            # batch tensor, which makes .view() raise on CPU (the cross-device .to('cuda')
            # copy happens to make it contiguous, which is why this only bit CPU runs).
            flat_logits = logits.reshape(B * T, C)
            targets = targets.reshape(B * T).long()
            loss = F.cross_entropy(flat_logits, targets)
        return logits, loss

    def forward_embedding(self, idx: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass focusing on the embedding and attention blocks.

        Args:
            idx (torch.Tensor): Input token indices.

        Returns:
            tuple: Output after attention blocks and the residual.
        """
        x = self._pre_attn_pass(idx)
        residual = x
        for block in self.attn_blocks:
            x, residual = block.forward_embedding(x)
        return x, residual

    def generate(self, idx: torch.Tensor, max_new_tokens: int) -> torch.Tensor:
        """
        Generates new tokens given a starting sequence.

        Args:
            idx (torch.Tensor): Initial sequence of token indices.
            max_new_tokens (int): Number of tokens to generate.

        Returns:
            torch.Tensor: The extended sequence of tokens.
        """
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.context_length:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


## Laboratorium interaktif: jalankan sel satu per satu

Semua definisi model tersedia langsung di sel sebelumnya.
Sel di bagian ini benar-benar menjalankan model di kernel, tanpa unduhan dataset.
Setiap sel kode menampilkan hasil yang dapat diperiksa. Model sengaja kecil dan
menggunakan tokenizer karakter lokal, bukan tokenizer GPT-2 untuk training penuh.
Jalankan berurutan. Untuk mengulang eksperimen dari awal, restart kernel lalu Run all.
Variabel `model`, `xb`, `yb`, `logits`, `loss`, dan `optimizer` tetap tersedia.


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `kernel` | `DEVICE` | Konfigurasi demo: string varian, string perangkat cpu, dan integer seed 1337 untuk generator acak. |
| `kernel` | `F` | Fungsi tensor neural network tanpa objek layer, misalnya linear dan cross entropy. |
| `kernel` | `KIND` | Konfigurasi demo: string varian, string perangkat cpu, dan integer seed 1337 untuk generator acak. |
| `kernel` | `SEED` | Konfigurasi demo: string varian, string perangkat cpu, dan integer seed 1337 untuk generator acak. |
| `kernel` | `TEXT` | String korpus kecil dalam RAM; seluruh kosakata berasal dari karakter di sini. |
| `kernel` | `cfg_demo` | Dictionary dimensi model demo: 2 head, C=16, konteks 12, satu blok, V sesuai vocabulary. |
| `kernel` | `char` | String satu karakter saat membangun vocabulary atau mengodekan teks. |
| `kernel` | `copy` | Modul penyalin objek; deepcopy membuat parameter pembanding independen. |
| `kernel` | `index` | Integer indeks saat enumerasi; i juga dipakai sebagai ID token saat decoding demo. |
| `kernel` | `itos` | Dictionary karakter ke ID (stoi) dan ID ke karakter (itos). |
| `kernel` | `stoi` | Dictionary karakter ke ID (stoi) dan ID ke karakter (itos). |
| `kernel` | `torch` | Modul PyTorch: tensor, diferensiasi otomatis, dan optimizer. |
| `kernel` | `vocabulary` | List karakter unik yang diurutkan agar pemetaan ID konsisten. |

</details>


In [ ]:
import copy
import torch
import torch.nn.functional as F

KIND = 'baseline'
DEVICE = 'cpu'
SEED = 1337
torch.manual_seed(SEED)
torch.set_printoptions(precision=3, sci_mode=False)
TEXT = 'saya belajar model bahasa. model belajar menebak karakter berikutnya. '
vocabulary = sorted(set(TEXT))
stoi = {char: index for index, char in enumerate(vocabulary)}
itos = {index: char for char, index in stoi.items()}
cfg_demo = dict(n_head=2, n_embed=16, context_length=12,
                vocab_size=len(vocabulary), N_BLOCKS=1)
print('Eksperimen:', KIND, '| device:', DEVICE, '| PyTorch:', torch.__version__)
print('Konfigurasi:', cfg_demo)
print('Vocabulary:', stoi)


### Data: input dan target bergeser satu posisi

`xb[b, t]` adalah karakter saat ini, sedangkan `yb[b, t]` karakter setelahnya.
Bentuk `[B,T]` berarti jumlah contoh dan panjang konteks. Target bukan salinan input.


Contoh: teks `saya` memberi input `say` dan target `aya`. Tokenizer karakter memudahkan inspeksi; tokenizer subword dapat merepresentasikan potongan kata dan biasanya menghasilkan urutan yang lebih pendek.

<details><summary>Latihan: mengapa panjang input 12 memerlukan 13 karakter?</summary>

Target terakhir adalah karakter setelah input terakhir. Dua slice sepanjang 12 harus diambil dari urutan sepanjang 13; target sudah bergeser, jadi forward model tidak boleh menggesernya lagi.
</details>


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `kernel` | `char` | String satu karakter saat membangun vocabulary atau mengodekan teks. |
| `kernel` | `encoded` | Tensor integer satu dimensi berisi semua ID karakter korpus. |
| `kernel` | `i` | Integer indeks saat enumerasi; i juga dipakai sebagai ID token saat decoding demo. |
| `kernel` | `samples` | Tensor integer [2,13] sebelum dipisah menjadi input dan target. |
| `kernel` | `start` | Integer batas slice data atau batas bagian hidden milik expert; end eksklusif. |
| `kernel` | `xb` | Tensor integer [2,12]; xb input dan yb target yang bergeser satu karakter. |
| `kernel` | `yb` | Tensor integer [2,12]; xb input dan yb target yang bergeser satu karakter. |

</details>


In [ ]:
encoded = torch.tensor([stoi[char] for char in TEXT], dtype=torch.long, device=DEVICE)
samples = torch.stack([encoded[start:start + 13] for start in (0, 13)])
xb, yb = samples[:, :-1], samples[:, 1:]
assert xb.shape == yb.shape == (2, 12)
assert torch.equal(xb[:, 1:], yb[:, :-1])
print('xb:', xb, '\nyb:', yb)
print('Input :', ''.join(itos[i] for i in xb[0].tolist()))
print('Target:', ''.join(itos[i] for i in yb[0].tolist()))
print('Shape:', tuple(xb.shape), '| dtype:', xb.dtype)


### Model dan embedding

Embedding token ditambah embedding posisi menghasilkan `[B,T,C]`.
Salinan dense disimpan untuk membandingkan output sebelum training pada input yang sama.
Konversi sparse mengganti MLP; attention tetap dense.


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `kernel` | `dense_model` | Objek Transformer yang dilatih atau salinan dense pembanding sebelum training. |
| `kernel` | `model` | Objek Transformer yang dilatih atau salinan dense pembanding sebelum training. |
| `kernel` | `p` | Tensor parameter model; .grad berisi turunan loss setelah backward. |

</details>


In [ ]:
dense_model = Transformer(**cfg_demo).to(DEVICE)
model = copy.deepcopy(dense_model)
print('Parameter dense:', sum(p.numel() for p in dense_model.parameters()))
print(dense_model)


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `kernel` | `attention` | Tensor bobot attention [B,T,T], nol di masa depan dan jumlah baris satu setelah softmax. |
| `kernel` | `embedding` | Tensor [B,T,C] setelah penjumlahan embedding atau setelah LayerNorm sebelum attention. |
| `kernel` | `head` | Objek head attention pertama yang dibuka untuk inspeksi. |
| `kernel` | `k` | Tensor key/query/value [B,T,d] pada attention; k atribut TopKMLP adalah jumlah neuron terpilih. |
| `kernel` | `normalized` | Tensor [B,T,C] setelah penjumlahan embedding atau setelah LayerNorm sebelum attention. |
| `kernel` | `q` | Tensor key/query/value [B,T,d] pada attention; k atribut TopKMLP adalah jumlah neuron terpilih. |
| `kernel` | `scores` | Skor dot product [B,T,T] pada attention atau skor expert terpilih [B*T,k] pada MoE. |
| `kernel` | `v` | Tensor key/query/value [B,T,d] pada attention; k atribut TopKMLP adalah jumlah neuron terpilih. |

</details>


In [ ]:
with torch.no_grad():
    embedding = model._pre_attn_pass(xb)
    normalized = model.attn_blocks[0].ln1(embedding)
    head = model.attn_blocks[0].attn.heads[0]
    q, k, v = head.query(normalized), head.key(normalized), head.value(normalized)
    scores = q @ k.transpose(-2, -1) / (q.shape[-1] ** 0.5)
    attention = scores.masked_fill(head.tril[:12, :12] == 0, float('-inf')).softmax(-1)
assert torch.allclose(attention.sum(-1), torch.ones_like(attention.sum(-1)))
assert torch.count_nonzero(attention.triu(1)) == 0
print('Embedding:', tuple(embedding.shape), '| Q/K/V:', tuple(q.shape))
print('Attention contoh pertama (baris=query, kolom=key):\n', attention[0])
print('Jumlah tiap baris:', attention[0].sum(-1))


Matriks attention berbentuk segitiga: posisi awal tidak boleh melihat token masa depan.
Softmax membuat jumlah setiap baris menjadi 1. Sel berikut memeriksa MLP setelah
attention dan residual, tepat pada input yang dipakai blok asli.


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `kernel` | `block` | Objek blok Transformer saat iterasi atau blok pertama dalam demo. |
| `kernel` | `mlp` | Objek MLP blok, dense atau varian sparse sesuai notebook. |
| `kernel` | `mlp_input` | Tensor [B,T,C] sebelum/sesudah MLP dalam jalur residual blok. |
| `kernel` | `mlp_output` | Tensor [B,T,C] sebelum/sesudah MLP dalam jalur residual blok. |
| `kernel` | `residual` | Tensor [B,T,C] jalur residual yang dijumlahkan dengan keluaran sublayer. |

</details>


In [ ]:
with torch.no_grad():
    block = model.attn_blocks[0]
    residual = embedding + block.attn(block.ln1(embedding))
    mlp_input = block.ln2(residual)
    mlp_output = block.mlp(mlp_input)
mlp = block.mlp
print('MLP input/output:', tuple(mlp_input.shape), tuple(mlp_output.shape))
print('Contoh output:', mlp_output[0, 0, :8])


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `kernel` | `activations` | Tensor aktivasi hidden [B,T,4C] setelah ReLU. |

</details>


In [ ]:
with torch.no_grad():
    activations = mlp.relu(mlp.hidden(mlp_input))
print('Hidden:', tuple(activations.shape))
print('Fraksi nol setelah ReLU:', (activations == 0).float().mean().item())
print('Nol dari ReLU bukan mask pruning; seluruh matriks tetap dihitung.')


### Forward dan loss

`logits` mempunyai bentuk `[B,T,V]`: skor setiap token vocabulary pada tiap posisi.
Cross entropy membandingkan skor dengan target berikutnya. MoE menambahkan loss
penyeimbang router; bandingkan cross entropy terpisah dari total loss.


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `kernel` | `aux_loss` | Tensor skalar penalti keseimbangan router; nol untuk demo non-MoE, None sebelum forward pertama MoEMLP. |
| `kernel` | `b` | Objek blok Transformer saat iterasi atau blok pertama dalam demo. |
| `kernel` | `ce_loss` | Tensor skalar objective; nama awalan menunjukkan CE, pembanding dense, SFT, reward, DPO, PPO, atau GRPO. |
| `kernel` | `dense_logits` | Tensor [B,T,V] dari model pembanding dense atau input contoh SFT. |
| `kernel` | `dense_loss` | Tensor skalar objective; nama awalan menunjukkan CE, pembanding dense, SFT, reward, DPO, PPO, atau GRPO. |
| `kernel` | `logits` | Skor sebelum softmax [B,T,V]; dalam DPO berupa selisih log-ratio [B], saat generasi diiris menjadi [B,V]. |
| `kernel` | `loss` | Tensor skalar objective; nama awalan menunjukkan CE, pembanding dense, SFT, reward, DPO, PPO, atau GRPO. |

</details>


In [ ]:
model.train()
logits, ce_loss = model(xb, yb)
aux_loss = (sum(b.mlp.aux_loss for b in model.attn_blocks)
            if KIND == 'moe' else ce_loss.new_zeros(()))
loss = ce_loss + 0.01 * aux_loss
with torch.no_grad():
    dense_logits, dense_loss = dense_model(xb, yb)
assert logits.shape == (2, 12, len(vocabulary))
assert torch.isfinite(loss)
print('Logits:', tuple(logits.shape))
print('CE:', ce_loss.item(), '| aux:', aux_loss.item(), '| total:', loss.item())
print('Dense CE sebelum training:', dense_loss.item())
print('Selisih logits maksimum:', (logits.detach() - dense_logits).abs().max().item())


### Backward: lihat gradien sebelum bobot berubah

Sel ini menghitung ulang forward agar aman dijalankan ulang setelah backward.
`zero_grad` menghapus akumulasi lama. `backward` menghitung turunan; optimizer belum
mengubah parameter. Aktifkan `DEBUG_ANOMALY` untuk melacak operasi backward bermasalah.


Autograd menerapkan aturan rantai pada grafik operasi: jika $L=L(z)$ dan $z=f(\theta)$, maka $\nabla_\theta L=J_f(\theta)^\top\nabla_zL$. Parameter $\theta$ adalah bobot terlatih. `backward()` mengisi `.grad`; `optimizer.step()` baru mengubah bobot. AdamW memakai rata-rata bergerak gradien dan kuadrat gradien, serta weight decay terpisah. Ini lebih adaptif daripada langkah SGD $\theta\leftarrow\theta-\eta\nabla_\theta L$; $\eta$ adalah learning rate. [Autograd PyTorch](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html).

<details><summary>Latihan: apakah grad norm nol berarti kodenya rusak?</summary>

Belum tentu. Mask pruning, ReLU yang mati, atau expert yang tidak terpilih dapat membuat sebagian gradien nol. Yang diperiksa adalah gradien finite, loss terhitung, dan ada bobot yang berubah. `NaN` dan tak hingga berbeda dari nol.
</details>


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `kernel` | `DEBUG_ANOMALY` | Boolean untuk pelacakan operasi yang menyebabkan backward bermasalah. |
| `kernel` | `aux_loss` | Tensor skalar penalti keseimbangan router; nol untuk demo non-MoE, None sebelum forward pertama MoEMLP. |
| `kernel` | `b` | Objek blok Transformer saat iterasi atau blok pertama dalam demo. |
| `kernel` | `before_update` | Salinan tensor embedding sebelum optimizer.step, tanpa hubungan autograd. |
| `kernel` | `ce_loss` | Tensor skalar objective; nama awalan menunjukkan CE, pembanding dense, SFT, reward, DPO, PPO, atau GRPO. |
| `kernel` | `logits` | Skor sebelum softmax [B,T,V]; dalam DPO berupa selisih log-ratio [B], saat generasi diiris menjadi [B,V]. |
| `kernel` | `loss` | Tensor skalar objective; nama awalan menunjukkan CE, pembanding dense, SFT, reward, DPO, PPO, atau GRPO. |
| `kernel` | `name` | String nama parameter atau atribut mask dalam loop inspeksi. |
| `kernel` | `optimizer` | Objek AdamW dengan parameter, learning rate, dan momen gradien tersimpan di RAM. |
| `kernel` | `parameter` | Tensor parameter model; .grad berisi turunan loss setelah backward. |

</details>


In [ ]:
DEBUG_ANOMALY = False
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
optimizer.zero_grad(set_to_none=True)
with torch.autograd.set_detect_anomaly(DEBUG_ANOMALY):
    logits, ce_loss = model(xb, yb)
    aux_loss = (sum(b.mlp.aux_loss for b in model.attn_blocks)
                if KIND == 'moe' else ce_loss.new_zeros(()))
    loss = ce_loss + 0.01 * aux_loss
    loss.backward()
for name, parameter in model.named_parameters():
    if parameter.grad is not None:
        assert torch.isfinite(parameter.grad).all(), name
        print(name, '| grad norm:', round(parameter.grad.norm().item(), 6))
before_update = model.token_embed.weight.detach().clone()


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `kernel` | `delta` | Float maksimum perubahan absolut bobot embedding setelah satu langkah optimizer. |

</details>


In [ ]:
optimizer.step()
if KIND == 'evolutionary':
    enforce_model_masks(model)
delta = (model.token_embed.weight.detach() - before_update).abs().max().item()
print('Perubahan maksimum embedding:', delta)
print('Jalankan ulang sel backward sebelum mengulang sel optimizer ini.')


### Latihan pendek: menghafal satu batch

Loss per langkah ditampilkan. Penurunan pada batch yang sama hanya menunjukkan
optimisasi bekerja, belum membuktikan generalisasi. Gunakan data validasi terpisah
untuk eksperimen sesungguhnya. SET melakukan rewiring setiap lima langkah di demo ini.


<details><summary>Latihan: apakah loss akhir lebih kecil membuktikan model memahami bahasa?</summary>

Tidak. Sepuluh langkah memakai batch yang sama, sehingga model dapat menghafal. Ukur loss pada teks terpisah untuk menilai generalisasi. Tidak ada jaminan loss turun pada setiap langkah, terutama ketika SET mengganti topologi.
</details>


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `kernel` | `aux_loss` | Tensor skalar penalti keseimbangan router; nol untuk demo non-MoE, None sebelum forward pertama MoEMLP. |
| `kernel` | `b` | Objek blok Transformer saat iterasi atau blok pertama dalam demo. |
| `kernel` | `ce_loss` | Tensor skalar objective; nama awalan menunjukkan CE, pembanding dense, SFT, reward, DPO, PPO, atau GRPO. |
| `kernel` | `history` | List float cross entropy setiap langkah; disimpan dalam RAM. |
| `kernel` | `logits` | Skor sebelum softmax [B,T,V]; dalam DPO berupa selisih log-ratio [B], saat generasi diiris menjadi [B,V]. |
| `kernel` | `loss` | Tensor skalar objective; nama awalan menunjukkan CE, pembanding dense, SFT, reward, DPO, PPO, atau GRPO. |
| `kernel` | `step` | Integer nomor langkah training dari 1 sampai 10. |

</details>


In [ ]:
history = []
for step in range(1, 11):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    logits, ce_loss = model(xb, yb)
    aux_loss = (sum(b.mlp.aux_loss for b in model.attn_blocks)
                if KIND == 'moe' else ce_loss.new_zeros(()))
    loss = ce_loss + 0.01 * aux_loss
    assert torch.isfinite(loss), f'Loss tidak finite pada step {step}'
    loss.backward()
    optimizer.step()
    if KIND == 'evolutionary':
        enforce_model_masks(model)
        if step % 5 == 0:
            rewire_model(model, optimizer, fraction=0.3)
    history.append(ce_loss.item())
    print(f'step={step:02d} CE={ce_loss.item():.4f} total={loss.item():.4f}')
print('CE awal/akhir:', history[0], history[-1])


### Generasi

Model mengambil distribusi token pada posisi terakhir, memilih satu token, lalu
mengulang dengan konteks yang diperpanjang. Model kecil ini belum menghasilkan
bahasa yang baik. Output berubah jika seed atau bobot berubah.


<details><summary>Latihan: mengapa bentuk keluaran menjadi [1,28]?</summary>

Ada satu prompt dengan empat karakter. `generate` menambahkan 24 ID sehingga panjang akhirnya 28. Setiap prediksi hanya melihat paling banyak 12 ID terakhir; hasil yang dikembalikan tetap memuat seluruh urutan.
</details>


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `kernel` | `generated` | Tensor integer [1,28]: empat token prompt ditambah 24 token hasil sampling. |
| `kernel` | `i` | Integer indeks saat enumerasi; i juga dipakai sebagai ID token saat decoding demo. |

</details>


In [ ]:
model.eval()
torch.manual_seed(SEED)
with torch.no_grad():
    generated = model.generate(xb[:1, :4], max_new_tokens=24)
print('Token:', generated[0].tolist())
print('Teks:', ''.join(itos[i] for i in generated[0].tolist()))


### Debugging di notebook

- Setelah exception, jalankan `%debug` di sel baru. Gunakan `p xb.shape`, `p loss`,
  `up`, `down`, `n` (next), `s` (step), `c` (continue), dan `q` (keluar).
- Tambahkan `breakpoint()` tepat sebelum baris yang ingin diperiksa. Jangan
  meninggalkannya aktif ketika memilih Run all karena kernel menunggu input.
- Mengulang sel konfigurasi/model mereset bobot; mengulang loop melanjutkan training.

Dokumentasi: [IPython debugging dan magic](https://ipython.readthedocs.io/en/stable/interactive/magics.html).


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |

</details>


In [ ]:
print('Kelas model dalam kernel:', type(model).__name__)
print('Kelas MLP dalam kernel:', type(model.attn_blocks[0].mlp).__name__)
print('Variabel tersedia: model, xb, yb, logits, loss, optimizer, history, attention')
print('Untuk traceback interaktif setelah error: %debug')


### Membaca `masked_mean`

Rata-rata bermask: $\bar x=\sum_i m_ix_i/\max(1,\sum_i m_i)$. Mask memilih posisi yang relevan. Pembagi minimal satu membuat mask kosong menghasilkan nol, bukan pembagian nol.

<details><summary>Latihan membaca kode</summary>

Sebelum menjalankan, telusuri satu input sampai `return` (atau `yield`).
Apa yang tetap dan apa yang berubah pada bentuk tensor atau objeknya?
Cocokkan dengan tabel nama di bawah dan hasil demo setelah definisi model.
</details>


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `masked_mean` | `count` | Integer jumlah elemen yang dipilih pada pruning/rewire; pada masked_mean tensor skalar jumlah posisi aktif, minimal satu. |
| `masked_mean` | `mask` | Tensor mask: pada MLP memilih bobot, pada objective memilih token; bentuk mengikuti tensor yang disaring. |
| `masked_mean` | `total` | Jumlah elemen bobot pada active_connections; jumlah nilai bermask tensor skalar pada masked_mean. |
| `masked_mean` | `values` | Tensor nilai top-k [...,k]; pada masked_mean tensor nilai yang hendak dirata-ratakan. |

| Fungsi atau kelas | Peran |
| :--- | :--- |
| `masked_mean` | Rata-rata bermask: $\bar x=\sum_i m_ix_i/\max(1,\sum_i m_i)$. |

</details>


In [ ]:
def masked_mean(values: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Mean of ``values`` over positions where ``mask`` is truthy (safe if mask empty)."""
    mask = mask.to(values.dtype)
    total = (values * mask).sum()
    count = mask.sum().clamp(min=1.0)
    return total / count


### Membaca `sft_loss`

Supervised Fine-Tuning (SFT) melatih model pada contoh prompt dan jawaban. $L_{SFT}=-\sum_t m_{t+1}\log p(x_{t+1}\mid x_{\le t})/\max(1,\sum_t m_{t+1})$. Mask $m$ mengaktifkan target jawaban saja. Fungsi melakukan shift sendiri; masukkan token asli, bukan `yb` yang sudah bergeser. Demo mask enam posisi hanyalah contoh, bukan parser percakapan.

<details><summary>Latihan membaca kode</summary>

Sebelum menjalankan, telusuri satu input sampai `return` (atau `yield`).
Apa yang tetap dan apa yang berubah pada bentuk tensor atau objeknya?
Cocokkan dengan tabel nama di bawah dan hasil demo setelah definisi model.
</details>


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `sft_loss` | `V` | Integer V: jumlah token berbeda yang dapat diprediksi. |
| `sft_loss` | `ce` | Tensor cross entropy per target, kemudian dikalikan mask; [B,T-1] dalam SFT. |
| `sft_loss` | `logits` | Skor sebelum softmax [B,T,V]; dalam DPO berupa selisih log-ratio [B], saat generasi diiris menjadi [B,V]. |
| `sft_loss` | `loss_mask` | Tensor mask token jawaban; hanya posisi aktif menyumbang objective. |
| `sft_loss` | `mask` | Tensor mask: pada MLP memilih bobot, pada objective memilih token; bentuk mengikuti tensor yang disaring. |
| `sft_loss` | `targets` | Tensor integer target next-token; [B,T] sebelum flatten atau [B*T] untuk cross entropy. |
| `sft_loss` | `tokens` | Tensor integer [B,T] berisi ID token; bukan embedding. |

| Fungsi atau kelas | Peran |
| :--- | :--- |
| `sft_loss` | Supervised Fine-Tuning (SFT) melatih model pada contoh prompt dan jawaban. |

</details>


In [ ]:
def sft_loss(logits: torch.Tensor, tokens: torch.Tensor, loss_mask: torch.Tensor) -> torch.Tensor:
    """
    Masked next-token cross-entropy.

    Args:
        logits:    (B, T, V) model logits for ``tokens``.
        tokens:    (B, T) packed token ids.
        loss_mask: (B, T) 1 on tokens that should be predicted (assistant completions).

    Returns:
        Scalar loss averaged over masked target positions.
    """
    # Predict token t+1 from position t (same shift the base model uses).
    logits = logits[:, :-1, :]
    targets = tokens[:, 1:]
    mask = loss_mask[:, 1:].to(logits.dtype)

    V = logits.size(-1)
    ce = F.cross_entropy(logits.reshape(-1, V).float(), targets.reshape(-1).long(), reduction="none")
    ce = ce.view(targets.shape) * mask
    return ce.sum() / mask.sum().clamp(min=1.0)


### Membaca `bradley_terry_loss`

Reward model (RM) memberi skor skalar untuk satu jawaban. Untuk pasangan yang disukai dan ditolak, $L_{RM}=-\operatorname{mean}\log\sigma(r_+-r_-)$. Sigmoid $\sigma$ mengubah selisih skor menjadi probabilitas preferensi. Contoh memakai skor sintetis; tidak membangun reward model baru.

<details><summary>Latihan membaca kode</summary>

Sebelum menjalankan, telusuri satu input sampai `return` (atau `yield`).
Apa yang tetap dan apa yang berubah pada bentuk tensor atau objeknya?
Cocokkan dengan tabel nama di bawah dan hasil demo setelah definisi model.
</details>


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `bradley_terry_loss` | `chosen_rewards` | Tensor [B] skor reward jawaban disukai/ditolak; demo memakai dua pasangan. |
| `bradley_terry_loss` | `rejected_rewards` | Tensor [B] skor reward jawaban disukai/ditolak; demo memakai dua pasangan. |

| Fungsi atau kelas | Peran |
| :--- | :--- |
| `bradley_terry_loss` | Reward model (RM) memberi skor skalar untuk satu jawaban. |

</details>


In [ ]:
def bradley_terry_loss(chosen_rewards: torch.Tensor, rejected_rewards: torch.Tensor) -> torch.Tensor:
    """Mean ``-log sigmoid(chosen - rejected)`` over a batch of preference pairs."""
    return -F.logsigmoid(chosen_rewards - rejected_rewards).mean()


### Membaca `preference_accuracy`

Hitung fraksi pasangan dengan skor chosen lebih besar daripada rejected. Skor yang sama dianggap belum benar. Metrik ini tidak dipakai sebagai loss karena perbandingan diskret tidak memberi gradien yang berguna.

<details><summary>Latihan membaca kode</summary>

Sebelum menjalankan, telusuri satu input sampai `return` (atau `yield`).
Apa yang tetap dan apa yang berubah pada bentuk tensor atau objeknya?
Cocokkan dengan tabel nama di bawah dan hasil demo setelah definisi model.
</details>


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `preference_accuracy` | `chosen_rewards` | Tensor [B] skor reward jawaban disukai/ditolak; demo memakai dua pasangan. |
| `preference_accuracy` | `rejected_rewards` | Tensor [B] skor reward jawaban disukai/ditolak; demo memakai dua pasangan. |

| Fungsi atau kelas | Peran |
| :--- | :--- |
| `preference_accuracy` | Hitung fraksi pasangan dengan skor chosen lebih besar daripada rejected. |

</details>


In [ ]:
def preference_accuracy(chosen_rewards: torch.Tensor, rejected_rewards: torch.Tensor) -> torch.Tensor:
    """Fraction of pairs where the model scores the chosen response higher."""
    return (chosen_rewards > rejected_rewards).float().mean()


### Membaca `dpo_loss`

Direct Preference Optimization (DPO) menggunakan pasangan jawaban tanpa melatih reward model terpisah. $L=-\mathbb E\log\sigma(\beta[(\log\pi_+-\log\pi_-)-(\log\pi_{ref,+}-\log\pi_{ref,-})])$. $\pi$ adalah policy yang dilatih, reference dibekukan, dan $\beta$ mengatur skala perbandingan. Log-probabilitas dijumlahkan sepanjang jawaban. Reward implisit yang dikembalikan hanya diagnostik, dilepas dari grafik gradien dengan `detach`. [Paper DPO](https://arxiv.org/abs/2305.18290).

<details><summary>Latihan membaca kode</summary>

Sebelum menjalankan, telusuri satu input sampai `return` (atau `yield`).
Apa yang tetap dan apa yang berubah pada bentuk tensor atau objeknya?
Cocokkan dengan tabel nama di bawah dan hasil demo setelah definisi model.
</details>


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `dpo_loss` | `beta` | Float pengali log-ratio pada objective DPO, default 0.1. |
| `dpo_loss` | `chosen_reward` | Tensor [B] reward implisit beta*(log policy - log reference), detached untuk diagnostik. |
| `dpo_loss` | `logits` | Skor sebelum softmax [B,T,V]; dalam DPO berupa selisih log-ratio [B], saat generasi diiris menjadi [B,V]. |
| `dpo_loss` | `loss` | Tensor skalar objective; nama awalan menunjukkan CE, pembanding dense, SFT, reward, DPO, PPO, atau GRPO. |
| `dpo_loss` | `pi_logratios` | Tensor [B] selisih log-probabilitas chosen dan rejected pada policy/reference. |
| `dpo_loss` | `policy_chosen_logps` | Tensor [B] jumlah log-probabilitas token jawaban; policy dilatih, ref pembanding tetap; chosen/rejected menandai preferensi. |
| `dpo_loss` | `policy_rejected_logps` | Tensor [B] jumlah log-probabilitas token jawaban; policy dilatih, ref pembanding tetap; chosen/rejected menandai preferensi. |
| `dpo_loss` | `ref_chosen_logps` | Tensor [B] jumlah log-probabilitas token jawaban; policy dilatih, ref pembanding tetap; chosen/rejected menandai preferensi. |
| `dpo_loss` | `ref_logratios` | Tensor [B] selisih log-probabilitas chosen dan rejected pada policy/reference. |
| `dpo_loss` | `ref_rejected_logps` | Tensor [B] jumlah log-probabilitas token jawaban; policy dilatih, ref pembanding tetap; chosen/rejected menandai preferensi. |
| `dpo_loss` | `rejected_reward` | Tensor [B] reward implisit beta*(log policy - log reference), detached untuk diagnostik. |

| Fungsi atau kelas | Peran |
| :--- | :--- |
| `dpo_loss` | Direct Preference Optimization (DPO) menggunakan pasangan jawaban tanpa melatih reward model terpisah. |

</details>


In [ ]:
def dpo_loss(
    policy_chosen_logps: torch.Tensor,
    policy_rejected_logps: torch.Tensor,
    ref_chosen_logps: torch.Tensor,
    ref_rejected_logps: torch.Tensor,
    beta: float = 0.1,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Standard DPO loss. Inputs are summed response log-probs (B,).

    Returns ``(loss, chosen_reward, rejected_reward)`` where the implicit rewards
    ``beta * (policy_logp - ref_logp)`` are detached diagnostics.
    """
    pi_logratios = policy_chosen_logps - policy_rejected_logps
    ref_logratios = ref_chosen_logps - ref_rejected_logps
    logits = pi_logratios - ref_logratios
    loss = -F.logsigmoid(beta * logits).mean()
    chosen_reward = beta * (policy_chosen_logps - ref_chosen_logps).detach()
    rejected_reward = beta * (policy_rejected_logps - ref_rejected_logps).detach()
    return loss, chosen_reward, rejected_reward


### Membaca `ppo_policy_loss`

Proximal Policy Optimization (PPO) membatasi insentif perubahan policy melalui $L=-\mathbb E_m[\min(rA,\operatorname{clip}(r,1-\epsilon,1+\epsilon)A)]$, dengan $r=\exp(\log\pi_{baru}-\log\pi_{lama})$. Advantage $A$ mengukur hasil relatif terhadap baseline; tanda positif mendorong aksi, negatif menguranginya. Clipping bekerja pada objective, bukan menjamin semua probabilitas tetap di interval itu. Demo memakai advantage grup agar hitungannya kecil; PPO lengkap biasanya memakai value model dan estimasi advantage. [Paper PPO](https://arxiv.org/abs/1707.06347).

<details><summary>Latihan membaca kode</summary>

Sebelum menjalankan, telusuri satu input sampai `return` (atau `yield`).
Apa yang tetap dan apa yang berubah pada bentuk tensor atau objeknya?
Cocokkan dengan tabel nama di bawah dan hasil demo setelah definisi model.
</details>


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `ppo_policy_loss` | `advantages` | Tensor advantage [B] pada GRPO, atau [B,1]/[B,L] pada PPO; nilai positif berarti lebih baik dari baseline. |
| `ppo_policy_loss` | `clip` | Float epsilon batas clipping rasio policy, default 0.2. |
| `ppo_policy_loss` | `clipped` | Tensor indikator rasio di luar interval clipping, dirata-ratakan sebagai diagnostik. |
| `ppo_policy_loss` | `loss` | Tensor skalar objective; nama awalan menunjukkan CE, pembanding dense, SFT, reward, DPO, PPO, atau GRPO. |
| `ppo_policy_loss` | `mask` | Tensor mask: pada MLP memilih bobot, pada objective memilih token; bentuk mengikuti tensor yang disaring. |
| `ppo_policy_loss` | `new_logp` | Tensor [B,L] log-probabilitas token aksi dari policy baru, pengumpul data lama, atau reference tetap. |
| `ppo_policy_loss` | `old_logp` | Tensor [B,L] log-probabilitas token aksi dari policy baru, pengumpul data lama, atau reference tetap. |
| `ppo_policy_loss` | `ratio` | Tensor [B,L] rasio probabilitas policy baru terhadap policy lama. |
| `ppo_policy_loss` | `surr1` | Tensor [B,L] surrogate tanpa clipping, dengan clipping, atau minimum keduanya. |
| `ppo_policy_loss` | `surr2` | Tensor [B,L] surrogate tanpa clipping, dengan clipping, atau minimum keduanya. |

| Fungsi atau kelas | Peran |
| :--- | :--- |
| `ppo_policy_loss` | Proximal Policy Optimization (PPO) membatasi insentif perubahan policy melalui $L=-\mathbb E_m[\min(rA,\operatorname{clip}(r,1-\epsilon,1+\epsilon)A)]$, dengan $r=\exp(\log\pi_{baru}-\log\pi_{lama})$. |

</details>


In [ ]:
def ppo_policy_loss(
    new_logp: torch.Tensor,
    old_logp: torch.Tensor,
    advantages: torch.Tensor,
    mask: torch.Tensor,
    clip: float = 0.2,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Clipped surrogate policy loss. Returns (loss, clip_fraction)."""
    ratio = torch.exp(new_logp - old_logp)
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1.0 - clip, 1.0 + clip) * advantages
    loss = -masked_mean(torch.min(surr1, surr2), mask)
    clipped = ((ratio - 1.0).abs() > clip).float()
    return loss, masked_mean(clipped, mask)


### Membaca `group_advantages`

Group Relative Policy Optimization (GRPO) membandingkan beberapa jawaban untuk prompt yang sama. $A_i=(r_i-\bar r)/(s+\varepsilon)$. Kode memakai simpangan baku sampel ($G-1$ pada penyebut varians), sehingga ukuran grup harus lebih dari satu. Reward identik menghasilkan advantage nol. Urutan input harus mengelompokkan jawaban per prompt.

<details><summary>Latihan membaca kode</summary>

Sebelum menjalankan, telusuri satu input sampai `return` (atau `yield`).
Apa yang tetap dan apa yang berubah pada bentuk tensor atau objeknya?
Cocokkan dengan tabel nama di bawah dan hasil demo setelah definisi model.
</details>


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `group_advantages` | `adv` | Tensor advantage yang dinormalisasi per grup atau diperluas menjadi [B,1] untuk broadcasting. |
| `group_advantages` | `eps` | Float konstanta kecil pada penyebut advantage untuk menghindari pembagian nol. |
| `group_advantages` | `group_size` | Integer jumlah jawaban per prompt; harus lebih dari satu untuk std sampel. |
| `group_advantages` | `mean` | Tensor [jumlah_prompt,1] rata-rata dan simpangan baku sampel reward per grup. |
| `group_advantages` | `r` | Tensor reward [jumlah_prompt,group_size] setelah reshape; bukan rasio policy. |
| `group_advantages` | `rewards` | Tensor satu dimensi reward, disusun per grup jawaban untuk prompt yang sama. |
| `group_advantages` | `std` | Tensor [jumlah_prompt,1] rata-rata dan simpangan baku sampel reward per grup. |

| Fungsi atau kelas | Peran |
| :--- | :--- |
| `group_advantages` | Group Relative Policy Optimization (GRPO) membandingkan beberapa jawaban untuk prompt yang sama. |

</details>


In [ ]:
def group_advantages(rewards: torch.Tensor, group_size: int, eps: float = 1e-4) -> torch.Tensor:
    """
    Group-relative advantage: ``(r - group_mean) / (group_std + eps)``.

    ``rewards`` is (num_prompts * group_size,) laid out group-contiguously (all G samples
    of prompt 0, then prompt 1, ...). Returns advantages of the same shape.
    """
    r = rewards.view(-1, group_size)
    mean = r.mean(dim=1, keepdim=True)
    std = r.std(dim=1, keepdim=True)
    adv = (r - mean) / (std + eps)
    return adv.reshape(-1)


### Membaca `k3_kl`

Untuk $d=\log\pi_{ref}-\log\pi_{baru}$, hitung $e^d-d-1\ge0$. Ini estimator penalti KL; kesetaraan ekspektasinya dengan KL memerlukan sampling dari policy baru dan dukungan distribusi yang sesuai. Angka sintetis demo hanya menunjukkan aljabar penalti.

<details><summary>Latihan membaca kode</summary>

Sebelum menjalankan, telusuri satu input sampai `return` (atau `yield`).
Apa yang tetap dan apa yang berubah pada bentuk tensor atau objeknya?
Cocokkan dengan tabel nama di bawah dan hasil demo setelah definisi model.
</details>


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `k3_kl` | `diff` | Tensor [B,L] selisih log-probabilitas reference minus policy baru. |
| `k3_kl` | `new_logp` | Tensor [B,L] log-probabilitas token aksi dari policy baru, pengumpul data lama, atau reference tetap. |
| `k3_kl` | `ref_logp` | Tensor [B,L] log-probabilitas token aksi dari policy baru, pengumpul data lama, atau reference tetap. |

| Fungsi atau kelas | Peran |
| :--- | :--- |
| `k3_kl` | Untuk $d=\log\pi_{ref}-\log\pi_{baru}$, hitung $e^d-d-1\ge0$. |

</details>


In [ ]:
def k3_kl(new_logp: torch.Tensor, ref_logp: torch.Tensor) -> torch.Tensor:
    """Per-token unbiased, non-negative KL estimator (Schulman's k3) for KL(policy||ref)."""
    diff = ref_logp - new_logp
    return torch.exp(diff) - diff - 1.0


### Membaca `grpo_loss`

GRPO menggabungkan surrogate terpotong dengan penalti reference: $L=-\mathbb E_m[\min(rA,\operatorname{clip}(r,1-\epsilon,1+\epsilon)A)-\lambda K]$. $K$ berasal dari `k3_kl`, $\lambda$ adalah `kl_coef`, dan satu advantage jawaban disiarkan ke token-token jawaban itu. Kode merata-ratakan token aktif seluruh batch. RLHF berarti Reinforcement Learning from Human Feedback; SFT, DPO, PPO, dan GRPO merupakan pilihan tahap/metode, bukan urutan wajib semuanya. [DeepSeekMath, pengenalan GRPO](https://arxiv.org/abs/2402.03300).

<details><summary>Latihan membaca kode</summary>

Sebelum menjalankan, telusuri satu input sampai `return` (atau `yield`).
Apa yang tetap dan apa yang berubah pada bentuk tensor atau objeknya?
Cocokkan dengan tabel nama di bawah dan hasil demo setelah definisi model.
</details>


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `grpo_loss` | `adv` | Tensor advantage yang dinormalisasi per grup atau diperluas menjadi [B,1] untuk broadcasting. |
| `grpo_loss` | `advantages` | Tensor advantage [B] pada GRPO, atau [B,1]/[B,L] pada PPO; nilai positif berarti lebih baik dari baseline. |
| `grpo_loss` | `clip` | Float epsilon batas clipping rasio policy, default 0.2. |
| `grpo_loss` | `kl` | Tensor [B,L] estimator penalti KL, satu nilai per token. |
| `grpo_loss` | `kl_coef` | Float lambda pengali penalti KL terhadap reference, default 0.04. |
| `grpo_loss` | `loss` | Tensor skalar objective; nama awalan menunjukkan CE, pembanding dense, SFT, reward, DPO, PPO, atau GRPO. |
| `grpo_loss` | `new_logp` | Tensor [B,L] log-probabilitas token aksi dari policy baru, pengumpul data lama, atau reference tetap. |
| `grpo_loss` | `old_logp` | Tensor [B,L] log-probabilitas token aksi dari policy baru, pengumpul data lama, atau reference tetap. |
| `grpo_loss` | `per_token` | Tensor [B,L] surrogate dikurangi penalti KL sebelum masked mean dan tanda negatif. |
| `grpo_loss` | `ratio` | Tensor [B,L] rasio probabilitas policy baru terhadap policy lama. |
| `grpo_loss` | `ref_logp` | Tensor [B,L] log-probabilitas token aksi dari policy baru, pengumpul data lama, atau reference tetap. |
| `grpo_loss` | `resp_mask` | Tensor mask token jawaban; hanya posisi aktif menyumbang objective. |
| `grpo_loss` | `stats` | Dictionary float statistik KL rata-rata dan fraksi rasio terpotong; bukan state optimizer. |
| `grpo_loss` | `surr1` | Tensor [B,L] surrogate tanpa clipping, dengan clipping, atau minimum keduanya. |
| `grpo_loss` | `surr2` | Tensor [B,L] surrogate tanpa clipping, dengan clipping, atau minimum keduanya. |
| `grpo_loss` | `surrogate` | Tensor [B,L] surrogate tanpa clipping, dengan clipping, atau minimum keduanya. |

| Fungsi atau kelas | Peran |
| :--- | :--- |
| `grpo_loss` | GRPO menggabungkan surrogate terpotong dengan penalti reference: $L=-\mathbb E_m[\min(rA,\operatorname{clip}(r,1-\epsilon,1+\epsilon)A)-\lambda K]$. |

</details>


In [ ]:
def grpo_loss(
    new_logp: torch.Tensor,
    old_logp: torch.Tensor,
    ref_logp: torch.Tensor,
    advantages: torch.Tensor,
    resp_mask: torch.Tensor,
    clip: float = 0.2,
    kl_coef: float = 0.04,
) -> tuple[torch.Tensor, dict]:
    """
    Token-level clipped surrogate + KL penalty.

    Args:
        new_logp/old_logp/ref_logp: (B, L) per-token log-probs (policy / sampling / ref).
        advantages: (B,) one scalar per completion, broadcast over its tokens.
        resp_mask:  (B, L) bool over response tokens.

    Returns:
        (loss, stats) with mean KL and clip fraction for logging.
    """
    adv = advantages[:, None]
    ratio = torch.exp(new_logp - old_logp)
    surr1 = ratio * adv
    surr2 = torch.clamp(ratio, 1.0 - clip, 1.0 + clip) * adv
    surrogate = torch.min(surr1, surr2)
    kl = k3_kl(new_logp, ref_logp)

    per_token = surrogate - kl_coef * kl
    loss = -masked_mean(per_token, resp_mask)
    stats = {
        "kl": masked_mean(kl, resp_mask).item(),
        "clipfrac": masked_mean(((ratio - 1.0).abs() > clip).float(), resp_mask).item(),
    }
    return loss, stats


### Dari pretraining ke SFT, reward, DPO, PPO, dan GRPO

Contoh berikut menjalankan fungsi objective asli pada tensor kecil. Ini demonstrasi
matematika loss, bukan training RLHF lengkap. Training dengan dataset besar tersedia melalui skrip di repositori.

SFT hanya menghitung loss pada target jawaban. Mask mengikuti posisi target setelah
shift. Ubah posisi bernilai satu untuk melihat bagian mana yang dilatih.


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `kernel` | `_` | Nilai atau indeks yang sengaja tidak digunakan. |
| `kernel` | `demo_logits` | Tensor [B,T,V] dari model pembanding dense atau input contoh SFT. |
| `kernel` | `response_mask` | Tensor mask token jawaban; hanya posisi aktif menyumbang objective. |
| `kernel` | `supervised_loss` | Tensor skalar objective; nama awalan menunjukkan CE, pembanding dense, SFT, reward, DPO, PPO, atau GRPO. |

</details>


In [ ]:
demo_logits, _ = model(xb)
response_mask = torch.zeros_like(xb)
response_mask[:, 6:] = 1
supervised_loss = sft_loss(demo_logits, xb, response_mask)
print('Mask jawaban:', response_mask[0])
print('Target aktif setelah shift:', int(response_mask[:, 1:].sum()))
print('SFT loss:', supervised_loss.item())


Reward model belajar mengurutkan dua jawaban. DPO langsung mengubah policy
berdasarkan selisih log-probabilitas chosen/rejected relatif terhadap reference.
Log-probabilitas di sini angka sintetis, bukan skor jawaban manusia.


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `kernel` | `chosen_implicit` | Tensor [B] reward implisit beta*(log policy - log reference), detached untuk diagnostik. |
| `kernel` | `chosen_rewards` | Tensor [B] skor reward jawaban disukai/ditolak; demo memakai dua pasangan. |
| `kernel` | `preference_dpo` | Tensor skalar objective; nama awalan menunjukkan CE, pembanding dense, SFT, reward, DPO, PPO, atau GRPO. |
| `kernel` | `preference_loss` | Tensor skalar objective; nama awalan menunjukkan CE, pembanding dense, SFT, reward, DPO, PPO, atau GRPO. |
| `kernel` | `rejected_implicit` | Tensor [B] reward implisit beta*(log policy - log reference), detached untuk diagnostik. |
| `kernel` | `rejected_rewards` | Tensor [B] skor reward jawaban disukai/ditolak; demo memakai dua pasangan. |

</details>


In [ ]:
chosen_rewards = torch.tensor([2.0, 0.5])
rejected_rewards = torch.tensor([0.0, 1.0])
preference_loss = bradley_terry_loss(chosen_rewards, rejected_rewards)
preference_dpo, chosen_implicit, rejected_implicit = dpo_loss(
    torch.tensor([-2.0, -3.0]), torch.tensor([-4.0, -2.0]),
    torch.tensor([-3.0, -3.0]), torch.tensor([-3.0, -3.0]),
)
print('Reward loss:', preference_loss.item())
print('Preference accuracy:', preference_accuracy(chosen_rewards, rejected_rewards).item())
print('DPO loss:', preference_dpo.item())
print('Implicit chosen/rejected rewards:', chosen_implicit, rejected_implicit)


PPO membandingkan policy baru dengan policy pengumpul rollout. Rasio terlalu
besar dibatasi dalam objective. GRPO menghitung advantage dari reward beberapa
jawaban untuk prompt yang sama, lalu memakai objective token dengan penalti KL.


<details><summary>AntiVibe: arti setiap nama dan fungsi dalam sel ini</summary>

Scope `kernel` bertahan antar sel; scope metode/fungsi lokal dibuat ulang saat dipanggil. Atribut `self` melekat pada objek. Ukuran simbolis mengikuti pengantar.

| Scope | Nama | Arti, tipe, dan bentuk |
| :--- | :--- | :--- |
| `kernel` | `action_mask` | Tensor mask token jawaban; hanya posisi aktif menyumbang objective. |
| `kernel` | `advantages` | Tensor advantage [B] pada GRPO, atau [B,1]/[B,L] pada PPO; nilai positif berarti lebih baik dari baseline. |
| `kernel` | `clip_fraction` | Tensor skalar fraksi token aktif dengan rasio keluar dari interval clipping. |
| `kernel` | `grpo_demo` | Tensor skalar objective; nama awalan menunjukkan CE, pembanding dense, SFT, reward, DPO, PPO, atau GRPO. |
| `kernel` | `grpo_stats` | Dictionary float statistik KL rata-rata dan fraksi rasio terpotong; bukan state optimizer. |
| `kernel` | `new_logp` | Tensor [B,L] log-probabilitas token aksi dari policy baru, pengumpul data lama, atau reference tetap. |
| `kernel` | `old_logp` | Tensor [B,L] log-probabilitas token aksi dari policy baru, pengumpul data lama, atau reference tetap. |
| `kernel` | `ppo_demo` | Tensor skalar objective; nama awalan menunjukkan CE, pembanding dense, SFT, reward, DPO, PPO, atau GRPO. |
| `kernel` | `rewards_demo` | Tensor satu dimensi reward, disusun per grup jawaban untuk prompt yang sama. |

</details>


In [ ]:
old_logp = torch.full((4, 3), -2.0)
new_logp = old_logp + torch.tensor([0.3, -0.1, 0.0, 0.1])[:, None]
action_mask = torch.ones_like(old_logp, dtype=torch.bool)
rewards_demo = torch.tensor([0.0, 1.0, 0.5, 1.5])
advantages = group_advantages(rewards_demo, group_size=4)
ppo_demo, clip_fraction = ppo_policy_loss(
    new_logp, old_logp, advantages[:, None], action_mask,
)
grpo_demo, grpo_stats = grpo_loss(
    new_logp, old_logp, old_logp, advantages, action_mask,
)
print('Reward grup:', rewards_demo, '| advantage:', advantages)
print('Rasio policy token pertama:', (new_logp - old_logp).exp()[:, 0])
print('PPO loss:', ppo_demo.item(), '| clip fraction:', clip_fraction.item())
print('GRPO loss:', grpo_demo.item(), '| stats:', grpo_stats)
